# SAC Navigation — segway_ed

Adapted from Arthur's SAC notebook (Ozzy's segway_2).

**Key changes vs segway_2:**

| | segway_2 (Ozzy) | segway_ed (yours) |
|---|---|---|
| Joints | freejoint + wheel hinge + reaction hinge | x_slide + body_pitch hinge |
| qpos | `[x, y, z, qw, qx, qy, qz, θ_w, θ_rw]` | `[x, θ]` (direct!) |
| qvel | `[vx,vy,vz,ωx,ωy,ωz, ω_w, ω_rw]` | `[ẋ, θ̇]` (direct!) |
| Actuators | wheel motor + reaction wheel | single `drive_motor` on x_slide |
| Control range | ±5 Nm | ±100 N |
| get_obs | quaternion → pitch via Rotation | `qpos[1]`, `qvel[1]` directly |
| Ground Z | lookup table (freejoint drops) | fixed at 0.18 m by XML |
| Arena | 5 × 5 m | 10 × 10 m (walls at ±5 m) |


In [ ]:
import mujoco
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os, imageio, base64, time, json
from IPython.display import display, HTML
from PIL import Image as PILImage, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── UPDATE THIS PATH ──────────────────────────────────────────────────────────
XML_PATH = "/path/to/your/segway_ed.xml"   # <-- set to your segway_ed.xml

SAVE_DIR     = "SavedSeeds"
TRAIN_SEEDS  = [42]
VERIFY_SEEDS = [788, 999, 555, 321, 444]
os.makedirs(SAVE_DIR, exist_ok=True)

# ── segway_ed model constants ─────────────────────────────────────────────────
# drive_motor acts on x_slide, ctrlrange = [-100, 100]
FORCE_MAX    = 100.0   # max drive force (N), matches ctrlrange in XML
X_NORM       =   5.0   # normalise x  (walls at ±5 m)
XD_NORM      =   5.0   # normalise xdot
TH_NORM      =   1.57  # normalise theta  (≈ π/2 rad)
THD_NORM     =   5.0   # normalise thetadot
X_DONE_LIMIT =   4.5   # |x| > this → episode ends  (wall contact at ~4.8 m)
TH_DONE      =   0.5   # |theta| > 0.5 rad (≈28.6°) → fallen


## Inline RunLogger

Replaces `shared_tracker.RunLogger` — keeps the notebook self-contained.
Swap back to `from shared_tracker import RunLogger` if preferred.


In [ ]:
class RunLogger:
    """Minimal training logger — drop-in for shared_tracker.RunLogger."""

    def __init__(self, algo, seed, log_every_steps=2000):
        self.algo = algo; self.seed = seed
        self.log_every_steps = log_every_steps
        self.rows = []
        self._ep_rewards = []; self._ep_arrives = []; self._ep_losses = []
        self._next_log   = log_every_steps
        self._t0         = time.time()

    def episode_end(self, total_steps, ep_count, ep_ret, arrived, loss):
        self._ep_rewards.append(ep_ret)
        self._ep_arrives.append(float(arrived))
        self._ep_losses.append(loss)
        avg = float(np.mean(self._ep_rewards[-100:]))
        rp  = float(np.mean(self._ep_arrives[-100:])) * 100.
        if total_steps >= self._next_log:
            self.rows.append({
                "steps":      total_steps,
                "avg_reward": avg,
                "loss":       float(np.mean(self._ep_losses[-100:])),
                "wall_time":  time.time() - self._t0,
            })
            self._next_log += self.log_every_steps
        return avg, rp

    def save(self, save_dir, tag, eval_results=None):
        payload = {"algo": self.algo, "seed": self.seed,
                   "rows": self.rows, "eval": eval_results,
                   "total_wall_time": self.total_wall_time}
        path = f"{save_dir}/{self.algo}_{tag}.json"
        with open(path, "w") as f:
            json.dump(payload, f, indent=2)
        print(f"Log saved → {path}")

    @property
    def total_wall_time(self): return time.time() - self._t0


## Observation helpers

segway_ed exposes pitch **directly** as a joint angle — no quaternion conversion.

| qpos | qvel |
|---|---|
| `[0]` x (x_slide position) | `[0]` ẋ |
| `[1]` θ (body_pitch angle, rad) | `[1]` θ̇ |


In [ ]:
def get_obs(data):
    """
    segway_ed state — no quaternion needed.
    Returns [x, xdot, theta, thetadot]  (same shape as segway_2 get_obs).
    """
    return np.array([data.qpos[0],   # x_slide position
                     data.qvel[0],   # x velocity
                     data.qpos[1],   # body_pitch angle
                     data.qvel[1]],  # pitch rate
                    dtype=np.float32)


def normalize_obs(obs):
    x, xd, th, thd = obs
    return np.array([
        np.clip(x   / X_NORM,   -1, 1),
        np.clip(xd  / XD_NORM,  -1, 1),
        np.clip(th  / TH_NORM,  -1, 1),
        np.clip(thd / THD_NORM, -1, 1),
    ], dtype=np.float32)


## Reward function

Identical to segway_2 — rewards progress, penalises tilt.


In [ ]:
def nav_reward(obs, prev_x, goal_x, at_goal, done, step):
    x, xd, theta, thd = obs
    if done:    return -20.0
    if at_goal: return  50.0
    r  = (abs(prev_x - goal_x) - abs(x - goal_x)) * 15.0  # progress reward
    r -= abs(x - goal_x) * 0.05                              # distance penalty
    r += 0.02                                                 # alive bonus
    if abs(theta) > 0.20:
        r -= (abs(theta) - 0.2) * 5.0                       # tilt penalty
    return float(r)


## SAC Networks

Architecture unchanged. Only `torque_max` is updated: **5 → 100 N**
(matching `ctrlrange` of `drive_motor` in segway_ed.xml).


In [ ]:
class NavActor(nn.Module):
    """
    Squashed Gaussian actor.
    Input : 5-dim (4 normalised obs + goal distance)
    Output: drive force in (-FORCE_MAX, FORCE_MAX) via tanh
    """
    LOG_STD_MIN = -5
    LOG_STD_MAX =  2

    def __init__(self, torque_max=FORCE_MAX):   # 100 N for segway_ed
        super().__init__()
        self.torque_max = torque_max
        self.net = nn.Sequential(
            nn.Linear(5, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
        )
        self.mean_layer    = nn.Linear(256, 1)
        self.log_std_layer = nn.Linear(256, 1)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, 0.5)
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.mean_layer.weight, 0.01)

    def forward(self, x):
        h = self.net(x)
        mean    = self.mean_layer(h)
        log_std = self.log_std_layer(h).clamp(self.LOG_STD_MIN, self.LOG_STD_MAX)
        return mean, log_std.exp()

    def get_action(self, obs, goal_x, deterministic=False):
        dist_norm = float(np.clip((goal_x - obs[0]) / 5., -1, 1))
        inp = torch.FloatTensor([*normalize_obs(obs), dist_norm]).unsqueeze(0)
        with torch.no_grad():
            mean, std = self(inp)
            if deterministic:
                raw  = mean
                logp = torch.zeros(1)
            else:
                dist = torch.distributions.Normal(mean, std)
                raw  = dist.rsample()
                logp = (dist.log_prob(raw)
                        - torch.log(1 - torch.tanh(raw).pow(2) + 1e-6)).sum(-1)
            force = torch.tanh(raw) * self.torque_max
        return force.item(), logp, None

    def sample(self, obs_t):
        mean, std = self(obs_t)
        dist  = torch.distributions.Normal(mean, std)
        raw   = dist.rsample()
        logp  = (dist.log_prob(raw)
                 - torch.log(1 - torch.tanh(raw).pow(2) + 1e-6)).sum(-1, keepdim=True)
        force = torch.tanh(raw) * self.torque_max
        return force, logp, torch.tanh(mean) * self.torque_max


class NavCritic(nn.Module):
    """Twin Q-networks Q1, Q2.  Input: 5-dim obs + 1-dim action = 6 total."""
    def __init__(self):
        super().__init__()
        def make_q():
            return nn.Sequential(
                nn.Linear(6, 256), nn.ReLU(),
                nn.Linear(256, 256), nn.ReLU(),
                nn.Linear(256, 1),
            )
        self.q1, self.q2 = make_q(), make_q()
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, 0.5)
                nn.init.zeros_(m.bias)

    def forward(self, obs_t, act_t):
        x = torch.cat([obs_t, act_t], dim=-1)
        return self.q1(x), self.q2(x)


## Replay Buffer

Unchanged — stores normalised 5-dim inputs.


In [ ]:
class ReplayBuffer:
    """Off-policy replay buffer. Stores normalised 5-dim inputs."""
    def __init__(self, capacity=1_000_000, obs_dim=5):
        self.cap  = capacity; self.ptr = self.size = 0
        self.obs  = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.act  = np.zeros((capacity, 1),       dtype=np.float32)
        self.rew  = np.zeros((capacity, 1),       dtype=np.float32)
        self.obs2 = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.done = np.zeros((capacity, 1),       dtype=np.float32)

    def push(self, o, a, r, o2, d):
        self.obs [self.ptr] = o;  self.act [self.ptr] = [a]
        self.rew [self.ptr] = r;  self.obs2[self.ptr] = o2
        self.done[self.ptr] = d
        self.ptr  = (self.ptr + 1) % self.cap
        self.size = min(self.size + 1, self.cap)

    def sample(self, n=256):
        i = np.random.randint(0, self.size, n)
        return (torch.FloatTensor(self.obs [i]),
                torch.FloatTensor(self.act [i]),
                torch.FloatTensor(self.rew [i]),
                torch.FloatTensor(self.obs2[i]),
                torch.FloatTensor(self.done[i]))

    def __len__(self): return self.size


## SAC `train_nav`

### Changes vs segway_2:

| | segway_2 | segway_ed |
|---|---|---|
| `reset_env` | sets qpos[2] (z), qpos[3:7] (quat), qvel[4] noise | sets qpos[1]=0 (upright), qvel[1] noise |
| `env_step` | `ctrl[0] = −u`, `ctrl[1] = 0.0` (2 actuators) | `ctrl[0] = u` (1 actuator, direct force) |
| Done `\|x\|` limit | 2.8 m | 4.5 m (walls at ±5 m) |
| Warmup random range | `Uniform(−5, 5)` | `Uniform(−100, 100)` |
| No ground-z lookup | freejoint needs it | z is fixed; not needed |

**Control sign:** `drive_motor` on `x_slide axis="1 0 0"` — positive force = +x direction. No negation (unlike segway_2's wheel motor).


In [ ]:
def train_nav(x_start=0.0, x_goal=2.0,
              seed=42, tag="nav_sac_ed",
              max_steps=3_000_000):
    torch.manual_seed(seed); np.random.seed(seed)

    GAMMA, TAU, LR   = 0.99, 0.005, 3e-4
    BATCH, WARMUP    = 256, 5000
    MAX_EP_STEPS     = 3000
    TARGET_ENT       = -1.0

    actor      = NavActor(torque_max=FORCE_MAX)
    critic     = NavCritic()
    critic_tgt = NavCritic(); critic_tgt.load_state_dict(critic.state_dict())
    for p in critic_tgt.parameters(): p.requires_grad = False

    actor_opt  = optim.Adam(actor.parameters(),  lr=LR)
    critic_opt = optim.Adam(critic.parameters(), lr=LR)
    log_alpha  = torch.tensor(0.0, requires_grad=True)
    alpha_opt  = optim.Adam([log_alpha], lr=LR)
    replay     = ReplayBuffer(1_000_000, obs_dim=5)

    model = mujoco.MjModel.from_xml_path(XML_PATH)
    data  = mujoco.MjData(model)

    # ── segway_ed reset ───────────────────────────────────────────────────────
    # No freejoint: z is fixed; qpos=[x, theta], qvel=[xdot, thetadot].
    def reset_env():
        mujoco.mj_resetData(model, data)
        data.qpos[0] = x_start
        data.qpos[1] = 0.0                              # upright (theta = 0)
        data.qvel[:] = 0.0
        data.qvel[1] = np.random.uniform(-0.01, 0.01)  # tiny tilt-rate perturbation
        mujoco.mj_forward(model, data)
        return get_obs(data)

    # ── segway_ed step ────────────────────────────────────────────────────────
    # Single actuator (drive_motor on x_slide). Positive force → +x direction.
    def env_step(force):
        u = float(np.clip(force, -FORCE_MAX, FORCE_MAX))
        data.ctrl[0] = u          # drive_motor; no negation, no ctrl[1]
        mujoco.mj_step(model, data)
        obs = get_obs(data)
        x, _, th, _ = obs
        done    = abs(x) > X_DONE_LIMIT or abs(th) > TH_DONE
        at_goal = abs(x - x_goal) < 0.10 and abs(th) < 0.25
        return obs, done, at_goal

    def make_inp(obs):
        return np.array([*normalize_obs(obs),
                         float(np.clip((x_goal - obs[0]) / 5., -1, 1))],
                        dtype=np.float32)

    # ── SAC update (unchanged) ────────────────────────────────────────────────
    def sac_update():
        if len(replay) < BATCH: return 0.0
        ob, ac, re, ob2, dn = replay.sample(BATCH)
        alpha = log_alpha.exp().detach()
        with torch.no_grad():
            na, nlp, _ = actor.sample(ob2)
            qt = torch.min(*critic_tgt(ob2, na)) - alpha * nlp
            y  = re + GAMMA * (1 - dn) * qt
        q1, q2 = critic(ob, ac)
        cl = nn.MSELoss()(q1, y) + nn.MSELoss()(q2, y)
        critic_opt.zero_grad(); cl.backward()
        nn.utils.clip_grad_norm_(critic.parameters(), 1.0); critic_opt.step()
        na, lp, _ = actor.sample(ob)
        al = (alpha * lp - torch.min(*critic(ob, na))).mean()
        actor_opt.zero_grad(); al.backward()
        nn.utils.clip_grad_norm_(actor.parameters(), 1.0); actor_opt.step()
        eal = -(log_alpha * (lp + TARGET_ENT).detach()).mean()
        alpha_opt.zero_grad(); eal.backward(); alpha_opt.step()
        with torch.no_grad():
            for p, pt in zip(critic.parameters(), critic_tgt.parameters()):
                pt.data.mul_(1 - TAU); pt.data.add_(TAU * p.data)
        return cl.item()

    # ── Training loop ─────────────────────────────────────────────────────────
    logger = RunLogger("SAC", seed, log_every_steps=2000)
    rewards = []; losses = []; best_avg = -9999; best_weights = None
    total_steps = ep_count = 0

    print(f"\nSAC | A={x_start}m → B={x_goal}m | budget={max_steps:,} steps")
    print(f"{'Steps':>9} | {'Ep':>5} | {'Avg100':>8} | {'Rec%':>6} | {'Alpha':>7} | {'CritL':>8}")
    print("-" * 64)

    while total_steps < max_steps:
        obs = reset_env(); ep_ret = 0.0; prev_x = obs[0]
        ep_step = 0; ep_cls = []; arrived_flag = False

        for _ in range(MAX_EP_STEPS):
            force = (np.random.uniform(-FORCE_MAX, FORCE_MAX) if total_steps < WARMUP
                     else actor.get_action(obs, x_goal)[0])
            obs2, done, at_goal = env_step(force)
            ep_step += 1; total_steps += 1
            r = nav_reward(obs2, prev_x, x_goal, at_goal, done, ep_step)
            ep_ret += r; prev_x = obs2[0]
            replay.push(make_inp(obs), force, r, make_inp(obs2), float(done or at_goal))
            if total_steps >= WARMUP: ep_cls.append(sac_update())
            obs = obs2
            if done or at_goal or ep_step >= MAX_EP_STEPS:
                arrived_flag = at_goal; break
            if total_steps >= max_steps: break

        ep_count += 1
        ep_loss = float(np.mean(ep_cls)) if ep_cls else 0.0
        rewards.append(ep_ret); losses.append(ep_loss)
        avg, rp = logger.episode_end(total_steps, ep_count, ep_ret, arrived_flag, ep_loss)

        if ep_count % 50 == 0:
            al   = float(log_alpha.exp().item())
            icon = "✅" if avg > 0 else "🔄" if avg > -20 else "⏳"
            print(f"{total_steps:>9,} | {ep_count:>5} | {avg:>8.2f} | "
                  f"{rp:>5.1f}% | {al:>7.4f} | {ep_loss:>8.3f} {icon}", flush=True)
            if avg > best_avg:
                best_avg = avg
                best_weights = {k: v.clone() for k, v in actor.state_dict().items()}
                torch.save(actor.state_dict(), f"{SAVE_DIR}/nav_sac_{tag}.pth")

    if best_weights: actor.load_state_dict(best_weights)
    return actor, rewards, losses, logger


In [ ]:
def train_one_verify_many(algo_label,
                          train_seed   = 42,
                          verify_seeds = (788, 999, 555, 321, 444),
                          x_start=0.0, x_goal=2.0,
                          step_budget=500_000):
    verify_seeds = list(verify_seeds)

    # ── 1. Train ──────────────────────────────────────────────────────────────
    print(f"\n{'='*60}\n  {algo_label}  TRAIN seed = {train_seed}\n{'='*60}")
    net, rewards, losses, logger = train_nav(
        x_start=x_start, x_goal=x_goal,
        seed=train_seed, tag=f"cmp_seed{train_seed}",
        max_steps=step_budget)

    # ── 2. Eval helper ────────────────────────────────────────────────────────
    def eval_policy(net, seed, n=20):
        torch.manual_seed(seed); np.random.seed(seed)
        model = mujoco.MjModel.from_xml_path(XML_PATH)
        data  = mujoco.MjData(model)
        strict = loose = fell = 0; max_xs = []; times = []
        for _ in range(n):
            mujoco.mj_resetData(model, data)
            data.qpos[0] = x_start; data.qpos[1] = 0.0
            data.qvel[:] = 0.0
            data.qvel[1] = np.random.uniform(-0.02, 0.02)
            mujoco.mj_forward(model, data)
            obs = get_obs(data); mx = 0.0; end = "timeout"
            for step in range(3000):
                with torch.no_grad():
                    f, _, _ = net.get_action(obs, x_goal, deterministic=True)
                data.ctrl[0] = float(np.clip(f, -FORCE_MAX, FORCE_MAX))
                mujoco.mj_step(model, data); obs = get_obs(data)
                x, _, th, _ = obs; mx = max(mx, x)
                if abs(x) > X_DONE_LIMIT or abs(th) > TH_DONE:
                    end = "fell"; break
                if abs(x - x_goal) < 0.10 and abs(th) < 0.35:
                    end = "strict"; times.append(step * model.opt.timestep); break
                elif abs(x - x_goal) < 0.25 and abs(th) < 0.35:
                    end = "loose";  times.append(step * model.opt.timestep)
            strict += end == "strict"; loose += end == "loose"; fell += end == "fell"
            max_xs.append(mx)
        return {"strict_pct": strict/n*100, "loose_pct": (strict+loose)/n*100,
                "fell_pct":   fell/n*100,   "avg_max_x": float(np.mean(max_xs)),
                "avg_time":   float(np.mean(times)) if times else 999.0}

    # ── 3. Verify on unseen seeds ─────────────────────────────────────────────
    print(f"\n  Verifying {algo_label} policy (trained on {train_seed}) "
          f"on {len(verify_seeds)} unseen seeds:")
    print(f"  {'Seed':>6} | {'Strict%':>8} | {'Loose%':>7} | {'Fell%':>6} | {'Time':>6}")
    print("  " + "-" * 48)
    per_seed = {}
    for vs in verify_seeds:
        r = eval_policy(net, vs); per_seed[vs] = r
        icon = "✅" if r["strict_pct"] >= 80 else "🔄" if r["strict_pct"] >= 50 else "❌"
        print(f"  {vs:>6} | {r['strict_pct']:>7.0f}% | {r['loose_pct']:>6.0f}% | "
              f"{r['fell_pct']:>5.0f}% | {r['avg_time']:>5.1f}s {icon}")

    # ── 4. Aggregate ──────────────────────────────────────────────────────────
    summary = {
        "train_seed":   train_seed,
        "verify_seeds": verify_seeds,
        "strict_pct":   float(np.mean([per_seed[v]["strict_pct"] for v in verify_seeds])),
        "strict_std":   float(np.std ([per_seed[v]["strict_pct"] for v in verify_seeds])),
        "fell_pct":     float(np.mean([per_seed[v]["fell_pct"]   for v in verify_seeds])),
        "avg_time":     float(np.mean([per_seed[v]["avg_time"] for v in verify_seeds
                                        if per_seed[v]["avg_time"] < 999] or [999])),
        "per_seed":     per_seed,
    }
    print(f"\n  {algo_label} verify avg: "
          f"{summary['strict_pct']:.1f}% ± {summary['strict_std']:.1f}%  strict  |  "
          f"{summary['fell_pct']:.1f}% fell")
    logger.save(SAVE_DIR, f"cmp_seed{train_seed}", eval_results=summary)
    return net, logger, summary


## Recording Function

Camera pulled back for the taller, wider segway_ed model.


In [ ]:
def record_all_seeds(nav_net,
                     train_seeds=[42],
                     verify_seeds=[788, 999, 555, 321, 444],
                     x_start=0.0, x_goal=2.0,
                     max_steps=3000):
    model    = mujoco.MjModel.from_xml_path(XML_PATH)
    data     = mujoco.MjData(model)
    dt       = model.opt.timestep
    renderer = mujoco.Renderer(model, height=480, width=640)
    cam = mujoco.MjvCamera()
    cam.type      = mujoco.mjtCamera.mjCAMERA_FREE
    cam.lookat    = np.array([1.0, 0.0, 0.5])  # aim at mid-body (segway_ed is taller)
    cam.distance  = 6.0                          # pulled back for larger arena
    cam.azimuth   = 90
    cam.elevation = -15
    PX_A = 160; PX_B = 480

    all_seeds = [(s, "TRAIN") for s in train_seeds] + [(s, "VERIFY") for s in verify_seeds]
    print("Recording seeds...")
    for seed, role in all_seeds:
        torch.manual_seed(seed); np.random.seed(seed)
        mujoco.mj_resetData(model, data)
        data.qpos[0] = x_start; data.qpos[1] = 0.0   # upright
        data.qvel[:] = 0.0; mujoco.mj_forward(model, data)
        obs = get_obs(data); frames = []; strict = arrived = fell = False

        for step in range(max_steps):
            with torch.no_grad():
                force, _, _ = nav_net.get_action(obs, x_goal, deterministic=True)
            data.ctrl[0] = float(np.clip(force, -FORCE_MAX, FORCE_MAX))
            mujoco.mj_step(model, data); obs = get_obs(data)
            x, _, theta, _ = obs
            fell    = abs(x) > X_DONE_LIMIT or abs(theta) > TH_DONE
            arrived = abs(x - x_goal) < 0.25 and abs(theta) < 0.35
            strict  = abs(x - x_goal) < 0.10 and abs(theta) < 0.35
            renderer.update_scene(data, camera=cam)
            img  = PILImage.fromarray(renderer.render())
            draw = ImageDraw.Draw(img); W, H = img.size
            prog = float(np.clip(x / x_goal, 0, 1)); bw = W - 40
            pcol = (0, 200, 0) if strict else (255, 140, 0) if arrived else (30, 100, 220)
            draw.rectangle([20, 8, W-20, 28], fill=(40, 40, 40))
            draw.rectangle([20, 8, 20 + int(bw * prog), 28], fill=pcol)
            draw.text((22, 10), "A", fill=(255, 255, 255))
            draw.text((W-28, 10), "B", fill=(255, 255, 255))
            draw.text((W//2-40, 10), f"{prog*100:.0f}%  x={x:.3f}m", fill=(255, 255, 255))
            badge_col = (0, 60, 140) if role == "TRAIN" else (100, 0, 140)
            draw.rectangle([8, 34, 170, 58], fill=badge_col)
            draw.text((12, 38), f"[{role}] seed={seed}", fill=(255, 255, 255))
            if strict:
                sc, st = (0, 120, 0),   f"✓ STRICT  x={x:.3f}m  t={step*dt:.1f}s"
            elif arrived:
                sc, st = (120, 100, 0), f"~ LOOSE   x={x:.3f}m  t={step*dt:.1f}s"
            elif fell:
                sc, st = (140, 0, 0),   f"✗ FELL    x={x:.3f}m  θ={np.degrees(theta):.1f}°"
            else:
                sc, st = (20, 20, 70),  (f"x={x:+.3f}m  θ={np.degrees(theta):+.1f}°  "
                                          f"u={force:+.1f}N  t={step*dt:.1f}s")
            draw.rectangle([175, 34, W-8, 58], fill=sc)
            draw.text((178, 38), st, fill=(255, 255, 255))
            frames.append(np.array(img))
            if fell or strict: break

        fname = f"{SAVE_DIR}/sac_{role.lower()}_seed{seed}.gif"
        imageio.mimsave(fname, frames, fps=30)
        status = "STRICT" if strict else "LOOSE" if arrived else "FELL" if fell else "TIMEOUT"
        print(f"  [{role}] seed={seed} → {status}   saved: {fname}")
    print("Done.")


## Plot Episode Traces


In [ ]:
def plot_episode_traces(net, algo="model", x_goal=2.0, save_prefix=None):
    save_prefix = save_prefix or algo.lower()
    model = mujoco.MjModel.from_xml_path(XML_PATH)
    data  = mujoco.MjData(model)
    dt    = model.opt.timestep
    mujoco.mj_resetData(model, data)
    data.qpos[0] = 0.0; data.qpos[1] = 0.0; data.qvel[:] = 0.0
    mujoco.mj_forward(model, data)
    obs = get_obs(data)
    ts, xs, ths, forces = [], [], [], []

    for step in range(3000):
        with torch.no_grad():
            force, _, _ = net.get_action(obs, x_goal, deterministic=True)
        u = float(np.clip(force, -FORCE_MAX, FORCE_MAX))
        data.ctrl[0] = u
        mujoco.mj_step(model, data); obs = get_obs(data)
        x, xd, th, thd = obs
        ts.append(step * dt); xs.append(x)
        ths.append(np.degrees(th)); forces.append(force)
        if (abs(x) > X_DONE_LIMIT or abs(th) > TH_DONE
                or (abs(x - x_goal) < 0.25 and abs(th) < 0.35)):
            break

    ts = np.array(ts); xs = np.array(xs)
    ths = np.array(ths); forces = np.array(forces)
    arrived = abs(xs[-1] - x_goal) < 0.25 and abs(ths[-1]) < 20
    rms = np.sqrt(np.mean(forces**2)); sat = np.abs(forces) > FORCE_MAX * 0.96

    fig = plt.figure(figsize=(15, 9))
    gs  = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.25, figure=fig)
    fig.suptitle(f"{algo} — Deterministic Episode Behaviour",
                 fontsize=13, fontweight="bold")

    ax = fig.add_subplot(gs[0, :])
    ax.plot(ts, xs, "#E74C3C", lw=2.5, label="x position")
    ax.axhline(0,      color="blue",  ls=":", lw=2, label="A (start)")
    ax.axhline(x_goal, color="green", ls=":", lw=2, label=f"B (goal={x_goal}m)")
    ax.axhspan(x_goal-0.25, x_goal+0.25, alpha=0.1, color="green", label="Goal zone ±0.25m")
    if arrived:
        idx = np.where(np.abs(xs - x_goal) < 0.25)[0][0]
        ax.axvline(ts[idx], color="green", ls="--", alpha=0.6)
        ax.annotate(f"ARRIVED\nt={ts[idx]:.1f}s", xy=(ts[idx], xs[idx]),
                    xytext=(ts[idx]+0.2, x_goal-0.3), color="green", fontsize=9,
                    arrowprops=dict(arrowstyle="->", color="green"))
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Position x (m)")
    ax.set_title("Position: A → B", fontweight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[1, 0])
    ax.plot(ts, ths, "#E74C3C", lw=2, label="θ (degrees)")
    ax.axhspan(-20, 20, alpha=0.06, color="green", label="Stable ±20°")
    ax.axhline(0, color="gray", alpha=0.4)
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Tilt Angle (°)")
    ax.set_title("Tilt Angle θ", fontweight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[1, 1])
    ax.plot(ts, forces, "#2ECC71", lw=2, label="Drive Force (N)")
    ax.fill_between(ts, forces, 0, where=(forces>0), alpha=0.15, color="red",  label="Forward")
    ax.fill_between(ts, forces, 0, where=(forces<0), alpha=0.15, color="blue", label="Backward")
    ax.axhline( FORCE_MAX, color="orange", ls=":", lw=1.5, label=f"Limit ±{FORCE_MAX:.0f}N")
    ax.axhline(-FORCE_MAX, color="orange", ls=":", lw=1.5)
    ax.axhline(0, color="gray", alpha=0.4)
    if sat.any():
        ax.fill_between(ts, -FORCE_MAX, FORCE_MAX, where=sat,
                        alpha=0.15, color="red", label="Saturated!")
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Drive Force (N)")
    ax.set_title(f"Drive Force  (RMS={rms:.2f} N)", fontweight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.savefig(f"{save_prefix}_episode_traces.png", dpi=150, bbox_inches="tight")
    plt.show(); print(f"Saved: {save_prefix}_episode_traces.png")

    print(f"\n{'='*55}\n  {algo} Episode Summary\n{'='*55}")
    print(f"  Duration   : {ts[-1]:.1f}s  ({len(ts)} steps)")
    print(f"  Max x      : {max(xs):.3f}m  / {x_goal}m")
    print(f"  Final x    : {xs[-1]:.3f}m")
    print(f"  Avg tilt   : {np.mean(np.abs(ths)):.1f}°")
    print(f"  Max tilt   : {max(np.abs(ths)):.1f}°")
    print(f"  Force RMS  : {rms:.3f} N")
    print(f"  Saturated  : {sat.sum()} / {len(ts)} steps ({sat.mean()*100:.0f}%)")
    print(f"{'='*55}")
    return {"max_x": float(max(xs)), "final_x": float(xs[-1]),
            "avg_tilt": float(np.mean(np.abs(ths))), "force_rms": float(rms),
            "arrived": bool(arrived), "duration_s": float(ts[-1])}


## Plot Loss and Reward

Unchanged — works with both logger and raw arrays.


In [ ]:
def plot_loss_reward(logger=None, rewards=None, losses=None,
                     algo="model", save_prefix=None):
    save_prefix = save_prefix or algo.lower()
    wall_time = None
    if logger is not None:
        steps_ts  = [r["steps"]      for r in logger.rows]
        reward_ts = [r["avg_reward"] for r in logger.rows]
        loss_ts   = [r["loss"]       for r in logger.rows]
        wall_time = getattr(logger, "total_wall_time", None)
        if wall_time is None and logger.rows:
            wall_time = logger.rows[-1].get("wall_time")
        have_steps = True
    else:
        if rewards is None or losses is None:
            raise ValueError("Pass either logger=, or both rewards= and losses=")
        have_steps = False

    fig, (axr, axl) = plt.subplots(1, 2, figsize=(15, 5))
    title = f"{algo} — Training"
    if wall_time is not None:
        title += f"   (wall time: {wall_time/60:.1f} min)"
    fig.suptitle(title, fontsize=13, fontweight="bold")

    if have_steps:
        x = np.array(steps_ts); y = np.array(reward_ts)
        axr.plot(x, y, color="#E74C3C", lw=2.5, label=f"avg reward (peak={y.max():.1f})")
        axr.set_xlabel("Environment steps")
    else:
        r_arr = np.array(rewards)
        w = min(100, max(2, len(r_arr)//10))
        roll = np.convolve(r_arr, np.ones(w)/w, mode="valid")
        axr.plot(np.arange(len(r_arr)), r_arr, color="#3498DB", alpha=0.2, lw=0.6,
                 label="per-episode")
        axr.plot(np.arange(len(roll))+w//2, roll, color="#E74C3C", lw=2.5,
                 label=f"{w}-ep avg (peak={roll.max():.1f})")
        axr.set_xlabel("Episode (no step data)")
    axr.axhline(0, color="green", ls="--", alpha=0.5)
    axr.set_ylabel("Reward"); axr.set_title("Training Reward", fontweight="bold")
    axr.legend(fontsize=9); axr.grid(alpha=0.3)

    loss_label = "Critic Loss" if algo.upper() == "SAC" else "PPO Loss"
    if have_steps:
        axl.plot(np.array(steps_ts), np.array(loss_ts), color="#9B59B6", lw=2, label="Loss")
        axl.set_xlabel("Environment steps")
    else:
        y = np.array(losses)
        axl.plot(np.arange(len(y)), y, color="#9B59B6", alpha=0.4, lw=0.6)
        k = min(20, max(2, len(y)//10))
        roll = np.convolve(y, np.ones(k)/k, mode="valid")
        axl.plot(np.arange(len(roll)), roll, color="#9B59B6", lw=2.5, label=f"smoothed ({k})")
        axl.set_xlabel("Update / rollout index")
    axl.axhline(0, color="green", ls="--", alpha=0.5)
    axl.set_ylabel(loss_label); axl.set_title("Training Loss", fontweight="bold")
    axl.legend(fontsize=9); axl.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{save_prefix}_loss_reward.png", dpi=150, bbox_inches="tight")
    plt.show(); print(f"Saved: {save_prefix}_loss_reward.png")
    if wall_time is not None:
        print(f"Training (wall) time: {wall_time/60:.1f} min  ({wall_time:.0f} s)")


## RUN ALL

### At 500,000 Steps


In [ ]:
net, logger, summary = train_one_verify_many("SAC", train_seed=42)


In [ ]:
record_all_seeds(net, train_seeds=[42], verify_seeds=[788, 999, 555], x_goal=2.0)


In [ ]:
plot_loss_reward(logger=logger, algo="SAC")


In [ ]:
plot_episode_traces(net, algo="SAC")
